# Prise en main du dataset et mise en place du dépôt Git

## Informations générales

| Élément | Détail |
|---|---|
| **Propriétaire** | Tiéba Bamba |
| **Projet** | Talk to my data |
| **Domaine** | Banque de détail - Recouvrement & Risque |
| **Étape** | 1/3 - mise en place du dépôt Git et EDA |
| **Notebook** | Setup du projet et analyse exploratoire des données (EDA) |
| **Date de création** | 24 août 2026 |
| **Statut** | Terminé |

## Contexte métier

La direction **« Recouvrement & Risque »** d'une banque de détail souhaite renforcer sa politique de relance et de recouvrement au prochain trimestre. Pour éclairer cette décision, nous allons étudier les données historiques de défaut de paiement de clients détenteurs d'une carte de crédit.

L'analyse doit permettre de :

1. Comprendre la structure et le contenu du dataset.
2. Identifier la variable cible et les principales variables explicatives.
3. Détecter les problèmes de qualité des données : valeurs manquantes, doublons, valeurs aberrantes et incohérences.
4. Décrire le profil des clients et les facteurs associés au défaut de paiement.
5. Préparer une base fiable pour les étapes ultérieures de modélisation et de mise à disposition d'un outil d'analyse conversationnelle.

## Objectifs de ce notebook

Ce notebook constitue la première étape du projet. Il a pour objectifs de :

- vérifier l'organisation initiale du projet et les chemins d'accès aux données ;
- charger le fichier de données brutes ;
- documenter les dimensions, les types et les modalités des variables ;
- réaliser une analyse exploratoire univariée et bivariée ;
- examiner la distribution de la variable cible et le déséquilibre éventuel des classes ;
- produire des visualisations utiles à la compréhension du risque ;
- formuler les premières observations et les décisions de préparation des données ;
- préparer et versionner les éléments nécessaires dans le dépôt Git.

## Déroulé annoncé

1. **Initialisation et vérification du projet** : imports, configuration et contrôle des chemins.
2. **Chargement des données** : lecture du dataset brut et aperçu des premières observations.
3. **Audit de qualité** : dimensions, types, valeurs manquantes, doublons, cardinalités et valeurs atypiques.
4. **Analyse exploratoire** : statistiques descriptives, distributions et relations avec la cible.
5. **Synthèse métier** : constats, limites, hypothèses et recommandations pour la suite.
6. **Préparation de la suite du projet** : éléments à conserver pour la modélisation et le dépôt Git.

## Résultat attendu

À la fin de ce notebook, nous disposerons d'une compréhension documentée du dataset, d'un premier diagnostic de qualité et d'une feuille de route claire pour la préparation des données et la modélisation.


# Exploration des données

In [25]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

## 1. Import des données

In [26]:
df= pd.read_csv("../data/raw/credit_card_default.csv")

print(f"Le jeu de données contient {df.shape[0]} lignes et {df.shape[1]} colonnes")
df.head()

Le jeu de données contient 2965 lignes et 26 colonnes


,id,limit_balance,sex,education_level,marital_status,age,pay_0,pay_2,pay_3,pay_4,...,bill_amt_5,bill_amt_6,pay_amt_1,pay_amt_2,pay_amt_3,pay_amt_4,pay_amt_5,pay_amt_6,default_payment_next_month,predicted_default_payment_next_month
0,27502.0,80000.0,1,6,1,54.0,0.0,0.0,0.0,0.0,...,26210.0,17643.0,2545.0,2208.0,1336.0,2232.0,542.0,348.0,1,"{\n ""predicted_default_payment_next_month"": [..."
1,26879.0,200000.0,1,4,1,49.0,0.0,0.0,0.0,0.0,...,50235.0,48984.0,1689.0,2164.0,2500.0,3480.0,2500.0,3000.0,0,"{\n ""predicted_default_payment_next_month"": [..."
2,18340.0,20000.0,2,6,2,22.0,0.0,0.0,0.0,0.0,...,500.0,0.0,4641.0,1019.0,900.0,0.0,1500.0,0.0,1,"{\n ""predicted_default_payment_next_month"": [..."
3,13692.0,260000.0,2,4,2,33.0,0.0,0.0,0.0,0.0,...,30767.0,29890.0,5000.0,5000.0,1137.0,5000.0,1085.0,5000.0,0,"{\n ""predicted_default_payment_next_month"": [..."
4,20405.0,150000.0,1,4,2,32.0,0.0,0.0,0.0,-1.0,...,143375.0,146411.0,4019.0,146896.0,157436.0,4600.0,4709.0,5600.0,0,"{\n ""predicted_default_payment_next_month"": [..."


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2965 entries, 0 to 2964
Data columns (total 26 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   id                                    2965 non-null   float64
 1   limit_balance                         2965 non-null   float64
 2   sex                                   2965 non-null   int64  
 3   education_level                       2965 non-null   int64  
 4   marital_status                        2965 non-null   int64  
 5   age                                   2965 non-null   float64
 6   pay_0                                 2965 non-null   float64
 7   pay_2                                 2965 non-null   float64
 8   pay_3                                 2965 non-null   float64
 9   pay_4                                 2965 non-null   float64
 10  pay_5                                 2965 non-null   int64  
 11  pay_6                       

## 2. Production du dictionnaire de données

Pour produire ce dictionnaire de données, j'ai effectué une recherche documentaire sur le Web afin de comprendre la signification des codes présents dans les variables catégorielles. Le fichier CSV fournit les noms des colonnes et leurs valeurs, mais ne précise pas la signification de chaque modalité. J'ai donc consulté la documentation du dataset de référence **Default of Credit Card Clients**, publiée par l'UCI Machine Learning Repository.

**Source consultée :** [UCI Machine Learning Repository - Default of Credit Card Clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)

Le tableau ci-dessous présente la définition de chaque variable, son type observé, un exemple issu du dataset et, lorsque cela est nécessaire, la signification de ses modalités. Les variables répétitives sont regroupées par famille afin de faciliter la lecture. Les codes non documentés sont signalés comme **non documenté / autre** plutôt que d'être interprétés sans preuve.

| Variable(s) | Définition | Type observé | Exemple | Modalités et signification |
|---|---|---|---|---|
| `id` | Identifiant unique du client | `float64` | `27502.0` | Identifiant, sans modalité métier |
| `limit_balance` | Montant du crédit accordé au client | `float64` | `80000.0` | Montant numérique continu |
| `sex` | Sexe du client | `int64` | `1` | `1` = homme ; `2` = femme |
| `education_level` | Niveau d'éducation du client | `int64` | `6` | `1` = études supérieures ; `2` = université ; `3` = lycée ; `4` = autre ; `0`, `5`, `6` = non documenté / autre |
| `marital_status` | Situation matrimoniale du client | `int64` | `1` | `1` = marié ; `2` = célibataire ; `3` = autre ; `0` = non documenté / autre |
| `age` | Âge du client | `float64` | `54.0` | Valeur numérique continue, exprimée en années |
| `pay_0`, `pay_2` à `pay_6` | Statut de remboursement observé chaque mois, du premier au dernier mois de la période étudiée | `float64` ou `int64` selon la colonne | `0.0` ou `0` | `-2` = aucun montant dû ou aucune consommation ; `-1` = paiement à temps ; `0` = crédit renouvelable utilisé sans retard déclaré ; `1` à `9` = retard de 1 à 9 mois ou plus |
| `bill_amt_1` à `bill_amt_6` | Montant de la facture mensuelle, du premier au dernier mois de la période étudiée | `float64` | `61454.0` pour `bill_amt_1` | Montant numérique continu |
| `pay_amt_1` à `pay_amt_6` | Montant payé chaque mois, du premier au dernier mois de la période étudiée | `float64` | `2545.0` pour `pay_amt_1` | Montant numérique continu |
| `default_payment_next_month` | Indique si le client fera défaut le mois suivant | `int64` | `1` | `0` = pas de défaut ; `1` = défaut de paiement |
| `predicted_default_payment_next_month` | Prédiction du défaut produite par le modèle | `str` | Réponse JSON du modèle | Texte au format JSON contenant la prédiction et le score associé |

**Lecture chronologique des suffixes :** dans le dataset, les variables de remboursement `pay_0`, `pay_2` à `pay_6` sont lues chronologiquement : `pay_0` correspond au premier mois observé, par exemple janvier, et `pay_6` au dernier mois observé, par exemple juin. Pour les familles `bill_amt_*` et `pay_amt_*`, le suffixe `1` correspond au premier mois observé et le suffixe `6` au dernier mois observé ; les suffixes intermédiaires représentent les mois successifs.

Pour les variables `pay_0` à `pay_6`, les valeurs négatives et la valeur `0` sont des codes de statut et non des montants. Les valeurs `1` à `9` correspondent à un nombre de mois de retard. Les modalités `education_level = 0, 5, 6` et `marital_status = 0` doivent rester identifiées comme non documentées ou autres, car leur signification précise n'est pas détaillée dans le fichier fourni.


#### Correction du type de certaines colonnes

nous remarquons que certaines colonnes devraient etre typées comme des variables catégoriellles nominales (aucune hiérarchie) mais on typées de la mauvaises mannières a savoir `sex` , `marital_status`, nous allons donc les changer en string


In [28]:
df[["sex","marital_status"]]= df[["sex","marital_status"]].astype("str")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2965 entries, 0 to 2964
Data columns (total 26 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   id                                    2965 non-null   float64
 1   limit_balance                         2965 non-null   float64
 2   sex                                   2965 non-null   str    
 3   education_level                       2965 non-null   int64  
 4   marital_status                        2965 non-null   str    
 5   age                                   2965 non-null   float64
 6   pay_0                                 2965 non-null   float64
 7   pay_2                                 2965 non-null   float64
 8   pay_3                                 2965 non-null   float64
 9   pay_4                                 2965 non-null   float64
 10  pay_5                                 2965 non-null   int64  
 11  pay_6                       

#### Vérification du déséquilibre des classes dans la colonne cible 

In [29]:
#repartition de la variable cible
df["default_payment_next_month"].value_counts(normalize=True)

default_payment_next_month
0    0.785835
1    0.214165
Name: proportion, dtype: float64

nous pouvons voir qu'ici nous avons un desequilibre des classes dans la variable cible , seulement environ 22% des cas sont des defauts de payments,nous allons donc tenir compte decette information dans les etapes suivantes 

### 2.1 Séapartion des données d'entrainement et de test

In [30]:
from sklearn.model_selection import train_test_split

In [31]:
#separation du jeu de données en train et test
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["default_payment_next_month"])

## 3.Analyse exploratoire 

### 3.1 Description des données


In [32]:
df_train.info()

<class 'pandas.DataFrame'>
Index: 2372 entries, 1479 to 2304
Data columns (total 26 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   id                                    2372 non-null   float64
 1   limit_balance                         2372 non-null   float64
 2   sex                                   2372 non-null   str    
 3   education_level                       2372 non-null   int64  
 4   marital_status                        2372 non-null   str    
 5   age                                   2372 non-null   float64
 6   pay_0                                 2372 non-null   float64
 7   pay_2                                 2372 non-null   float64
 8   pay_3                                 2372 non-null   float64
 9   pay_4                                 2372 non-null   float64
 10  pay_5                                 2372 non-null   int64  
 11  pay_6                         

La separation des données nous a généré 2372 lignes de données d'entrainement 


### 3.2 Statistitques univariés

**Variables numériques**

In [33]:
#variables numeriques
numeric_columns = df_train.select_dtypes(include=[np.number]).columns.tolist()
numeric_columns.remove("education_level")
numeric_columns.remove("id")

#description statistique des variables numeriques
df_train[numeric_columns].describe()

,limit_balance,age,pay_0,pay_2,pay_3,pay_4,pay_5,pay_6,bill_amt_1,bill_amt_2,...,bill_amt_4,bill_amt_5,bill_amt_6,pay_amt_1,pay_amt_2,pay_amt_3,pay_amt_4,pay_amt_5,pay_amt_6,default_payment_next_month
count,2372.000000,2372.000000,2372.000000,2372.000000,2372.000000,2372.000000,2372.000000,2372.000000,2372.000000,2372.000000,...,2372.000000,2372.000000,2372.000000,2372.000000,2.372000e+03,2372.000000,2372.000000,2372.000000,2372.000000,2372.000000
mean,163908.094435,35.195616,0.002951,-0.120573,-0.131956,-0.174115,-0.217960,-0.262226,52014.819140,50759.189713,...,44788.639123,41470.831366,39915.422428,6554.898398,6.547390e+03,5216.470067,4523.836425,4598.529933,5542.190978,0.214165
std,123563.482429,9.005440,1.114488,1.169253,1.159505,1.164839,1.143503,1.146924,70584.199692,69707.572807,...,61948.256095,58241.138744,56628.500034,22069.335095,3.168979e+04,14305.680346,13206.025538,14690.160349,18172.508206,0.410329
min,10000.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-5684.000000,-67526.000000,...,-27490.000000,-7220.000000,-51183.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,50000.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,4033.000000,3425.250000,...,2720.750000,2037.250000,1429.250000,1017.500000,1.000000e+03,537.500000,326.000000,383.500000,210.750000,0.000000
50%,140000.000000,34.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,25152.500000,23764.000000,...,20411.500000,19223.000000,18677.000000,2293.500000,2.275500e+03,2000.000000,1646.500000,1683.000000,1693.000000,0.000000
75%,230000.000000,41.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,70274.500000,67921.750000,...,59487.000000,53891.500000,52636.250000,5149.750000,5.038750e+03,4710.750000,4009.250000,4033.250000,4164.250000,0.000000
max,800000.000000,69.000000,8.000000,7.000000,7.000000,8.000000,7.000000,7.000000,533142.000000,512650.000000,...,485249.000000,441981.000000,424592.000000,493358.000000,1.227082e+06,199209.000000,202076.000000,300000.000000,403500.000000,1.000000


##### **Grandes tendances importantes à noter**

- Les montants de crédit et de facturation sont très hétérogènes :
    - `limit_balance`, `bill_amt_*` et `pay_amt_*` présentent des écarts très importants entre les valeurs faibles et les valeurs extrêmes.
    - La distribution est fortement asymétrique, avec un petit nombre de clients très exposés ou avec des montants très élevés.
    - Cela suggère la présence de valeurs atypiques à surveiller avant la modélisation.

- La plupart des clients sont en situation de remboursement normale ou quasi-normale :
    - Les variables `pay_*` sont majoritairement concentrées autour des valeurs `-1`, `0` et parfois `1` à `2`.
    - Cela montre qu’une majorité de clients paie à temps ou n’utilise pas de crédit renouvelable.
    - En revanche, une queue de distribution non négligeable montre des retards de paiement, ce qui est important pour le risque.

- Le comportement de remboursement varie fortement selon le mois :
    - Les séries `pay_0` à `pay_6` indiquent que les retards ne sont pas rares, mais restent concentrés dans des niveaux modérés pour beaucoup de clients.
    - Les cas de retard important (`3+` mois) sont moins fréquents, mais ils sont très révélateurs du risque.

- La variable cible est déséquilibrée :
    - La proportion de défaut est minoritaire, ce qui confirme le déséquilibre déjà observé dans les compte-rendus précédents.
    - Cette asymétrie doit être prise en compte lors du choix du modèle et lors de l’évaluation (métriques comme F1, Recall, PR-AUC, etc.).

- Les montants payés ne sont pas toujours alignés avec les factures :
    - Certaines lignes montrent des montants de paiement faibles ou nuls alors que les factures sont élevées.
    - Cela traduit une variabilité importante dans le comportement de paiement et peut être un bon indicateur de risque.

- La population est plutôt jeune à intermédiaire d’âge :
    - `age` est concentré dans la tranche des 20–50 ans.
    - La distribution reste assez large, mais la majorité des clients n’est pas très âgée, ce qui peut influencer la dynamique de remboursement.


En résumé, le signal principal du dataset est celui d’une population majoritairement stable financièrement, mais avec une minorité importante de clients présentant des retards de paiement et des montants de facturation très variables. C’est exactement le type de structure qui justifie une modélisation orientée risque et une préparation rigoureuse des données.

##### **Repérage des outliers**

In [34]:
from plotly.subplots import make_subplots

In [35]:
fig = make_subplots(
    rows=1,
    cols=1,
    subplot_titles=("Box plots de limit_balance et age",)
)

fig.add_trace(
    go.Box(
        y=df_train["limit_balance"],
        name="Limit balance",
        boxmean=True
    ),
    row=1,
    col=1
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Limit balance", "Âge"),
    shared_yaxes=False,
    horizontal_spacing=0.15
)

fig.add_trace(
    go.Box(
        y=df_train["limit_balance"],
        name="Limit balance",
        boxmean=True
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Box(
        y=df_train["age"],
        name="Âge",
        boxmean=True
    ),
    row=1,
    col=2
)

fig.update_layout(
    title="Détection des valeurs aberrantes",
    yaxis_title="Limit balance",
    yaxis2_title="Âge",
    showlegend=True,
    template="plotly_white"
)

fig.update_layout(
    title="Détection des valeurs aberrantes",
    yaxis_title="Valeurs",
    showlegend=True,
    template="plotly_white"
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

nous n'osbservons pas de valeurs aberrantes pour les variables `age` et `limit_balance`, certes ils il existe des valeurs en dehors de la boite mais celles ci restent plausibles 

In [36]:
bill_columns = [f"bill_amt_{i}" for i in range(1, 7)]

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=bill_columns,
    shared_yaxes=True,
    vertical_spacing=0.15,
    horizontal_spacing=0.08
)

for index, column in enumerate(bill_columns):
    row = index // 3 + 1
    col = index % 3 + 1

    fig.add_trace(
        go.Box(
            y=df_train[column],
            name=column,
            boxmean=True,
            showlegend=False
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title="Détection des valeurs aberrantes pour les montants des factures",
    template="plotly_white",
    height=700,
    showlegend=False
)

fig.update_yaxes(title_text="Montant de la facture", matches="y")

display(HTML(fig.to_html(include_plotlyjs="cdn")))

In [37]:
pay_columns = [f"pay_amt_{i}" for i in range(1, 7)]

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=pay_columns,
    shared_yaxes=True,
    vertical_spacing=0.15,
    horizontal_spacing=0.08
)

for index, column in enumerate(pay_columns):
    row = index // 3 + 1
    col = index % 3 + 1

    fig.add_trace(
        go.Box(
            y=df_train[column],
            name=column,
            boxmean=True,
            showlegend=False
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title="Détection des valeurs aberrantes pour les montants payés",
    template="plotly_white",
    height=700,
    showlegend=False
)
fig.update_yaxes(matches=None)
fig.update_yaxes(
    title_text="Montant payé",
    matches="y"
)

display(HTML(fig.to_html(include_plotlyjs="cdn")))

##### **Analyse des montants payés et des factures**

Les box plots mettent en évidence des valeurs atypiques dans les variables `pay_amt_*`, en particulier dans `pay_amt_2`. Un montant payé d’environ **1,23 million** apparaît très éloigné de la distribution générale. Cette valeur peut correspondre à une régularisation exceptionnelle, à une erreur de saisie ou à un remboursement anticipé. 

Concernant les variables `bill_amt_*`, les box plots ne révèlent pas nécessairement des valeurs aberrantes au sens statistique. Toutefois, certaines factures sont négatives. Ces montants peuvent correspondre à un avoir, une régularisation ou un solde créditeur résultant d’un paiement supérieur au montant dû. Leur signification exacte n’étant pas documentée dans le fichier, elles doivent être conservées dans un premier temps et faire l’objet d’une analyse complémentaire.

**Variables catégorielles**

In [38]:
categorical_variables = [
    ("sex", "Sexe"),
    ("marital_status", "Situation matrimoniale"),
    ("default_payment_next_month", "Défaut de paiement le mois suivant"),
    ("education_level", "Niveau d'éducation"),
]

fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=[title for _, title in categorical_variables],
    shared_yaxes=False,
    horizontal_spacing=0.08,
)

for col, (variable, title) in enumerate(categorical_variables, start=1):
    frequencies = df_train[variable].value_counts().sort_index()
    percentages = frequencies / frequencies.sum() * 100

    fig.add_trace(
        go.Bar(
            x=percentages.index.astype(str),
            y=percentages.values,
            text=[f"{value:.1f} %" for value in percentages.values],
            textposition="auto",
            name=variable,
            showlegend=False,
        ),
        row=1,
        col=col,
    )

    fig.update_xaxes(
        title_text="Modalité",
        row=1,
        col=col,
    )
    fig.update_yaxes(
        title_text="Fréquence (%)",
        row=1,
        col=col,
    )

fig.update_layout(
    title="Fréquence des modalités des variables catégorielles",
    template="plotly_white",
    height=500,
    width=1200,
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

Constats importants:
- nous remarquons que nous avons plus de femmes que d'hommes
- la modalité 0 représente un faible pourcentage dans la repartion de la variable `Marital_status` par conséquent celle ci sera regroupe avec la modalité 3 (autre)
- confirmation du déséquilibre des classes enoncé plus haut dans le notebook
- les modalités 4,5 et 6 représente un faible pourcentage dans la repartion de la variable `education_level`, etant donné que celle ci sont non documentés nous allons les regrouper dans la modalité 0 pour eviter des soucis de mauvais interpretations 

In [39]:
pay_status_columns = ["pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"]
fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=pay_status_columns,
    shared_yaxes=False,
    vertical_spacing=0.18,
    horizontal_spacing=0.08
)

for index, column in enumerate(pay_status_columns):
    row = index // 3 + 1
    col = index % 3 + 1
    frequencies = df_train[column].value_counts().sort_index()

    fig.add_trace(
        go.Bar(
            x=frequencies.index.astype(str),
            y=frequencies.values,
            text=frequencies.values,
            textposition="auto",
            showlegend=False
        ),
        row=row,
        col=col
    )

    fig.update_xaxes(
        title_text="Statut de remboursement",
        row=row,
        col=col
    )
    fig.update_yaxes(
        title_text="Effectif",
        row=row,
        col=col
    )

fig.update_layout(
    title="Répartition des statuts de remboursement",
    template="plotly_white",
    height=750,
    width=1200
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

On remarque que pour chaque variable de pay_* on a une concentration d'observations autour de  0 ce qui siginfie que la grande 
majorité des utilisateurs ont tendances a payer leurs factures avant la date limite

### 3.3 Statistitques bivariées

**defaut de paiement selon sex**

In [40]:
# Préparation des données pour le graphique
sex_labels = {
    "1": "Homme",
    "2": "Femme"
}

default_labels = {
    0: "Pas de défaut",
    1: "Défaut"
}

plot_data = df_train.copy()
plot_data["Sexe"] = plot_data["sex"].astype(str).map(sex_labels)
plot_data["Défaut"] = plot_data["default_payment_next_month"].map(default_labels)

counts = (
    plot_data.groupby(["Sexe", "Défaut"])
    .size()
    .reset_index(name="Effectif")
)

fig = go.Figure()

for default_status in ["Pas de défaut", "Défaut"]:
    data = counts[counts["Défaut"] == default_status]

    fig.add_trace(
        go.Bar(
            x=data["Sexe"],
            y=data["Effectif"],
            name=default_status,
            text=data["Effectif"],
            textposition="auto"
        )
    )

fig.update_layout(
    title="Défaut de paiement selon le sexe",
    xaxis_title="Sexe",
    yaxis_title="Effectif",
    barmode="group",
    template="plotly_white"
)

display(HTML(fig.to_html(include_plotlyjs="cdn")))

Le défaut de paiement concerne une minorité de clients, aussi bien chez les hommes que chez les femmes. Les proportions semblent relativement proches entre les deux groupes ; le sexe ne paraît donc pas être un facteur fortement discriminant dans cette première analyse.

**defaut de paiement selon le niveau d'éducation**

In [41]:
plot_data = df_train.copy()
plot_data["education_level_grouped"] = plot_data["education_level"].replace(
    {4: 0, 5: 0, 6: 0}
)

education_labels = {
    0: "Autre / non documenté",
    1: "Études supérieures",
    2: "Université",
    3: "Lycée",
}

default_labels = {
    0: "Pas de défaut",
    1: "Défaut de paiement",
}

plot_data["Niveau d'éducation"] = plot_data["education_level_grouped"].map(
    education_labels
)
plot_data["Défaut"] = plot_data["default_payment_next_month"].map(
    default_labels
)

education_order = [
    "Autre / non documenté",
    "Études supérieures",
    "Université",
    "Lycée",
]

default_order = ["Pas de défaut", "Défaut de paiement"]

counts = (
    plot_data.groupby(["Niveau d'éducation", "Défaut"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=education_order, columns=default_order, fill_value=0)
)

fig = go.Figure()

for default_status in default_order:
    fig.add_trace(
        go.Bar(
            x=counts.index,
            y=counts[default_status],
            name=default_status,
            text=counts[default_status],
            textposition="auto",
        )
    )

fig.update_layout(
    title="Défaut de paiement selon le niveau d'éducation",
    xaxis_title="Niveau d'éducation",
    yaxis_title="Effectif",
    barmode="group",
    template="plotly_white",
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

Les proportions de défaut varient selon le niveau d'éducation : elles sont d'environ 8 % pour la catégorie **Autre / non documenté**, 18 % pour les **études supérieures**, 24 % pour l'**université** et 24 % pour le **lycée**.

Dans cet échantillon, les clients ayant un niveau d'études supérieures semblent donc présenter une proportion de défaut légèrement plus faible. Toutefois, cette observation ne permet pas de conclure à une relation causale et doit être complétée par une analyse statistique.

**defaut de paiement selon le statut matrimonial**

In [42]:
plot_data = df_train.copy()

# Regroupement de la modalité 0 avec la modalité 3.
plot_data["marital_status_grouped"] = plot_data["marital_status"].replace(
    {"0": "3"}
)

marital_labels = {
    "1": "Marié",
    "2": "Célibataire",
    "3": "Autre / non documenté",
}

default_labels = {
    0: "Pas de défaut",
    1: "Défaut de paiement",
}

plot_data["Situation matrimoniale"] = plot_data["marital_status_grouped"].map(
    marital_labels
)
plot_data["Défaut"] = plot_data["default_payment_next_month"].map(
    default_labels
)

marital_order = [
    "Marié",
    "Célibataire",
    "Autre / non documenté",
]
default_order = ["Pas de défaut", "Défaut de paiement"]

counts = (
    plot_data.groupby(["Situation matrimoniale", "Défaut"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=marital_order, columns=default_order, fill_value=0)
)

fig = go.Figure()

for default_status in default_order:
    fig.add_trace(
        go.Bar(
            x=counts.index,
            y=counts[default_status],
            name=default_status,
            text=counts[default_status],
            textposition="auto",
        )
    )

fig.update_layout(
    title="Défaut de paiement selon la situation matrimoniale",
    xaxis_title="Situation matrimoniale",
    yaxis_title="Effectif",
    barmode="group",
    template="plotly_white",
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

le statut matrimoniale n'a aucune influence sur la probabilité d'avoir un defaut de crédit tout comme le sex


**defaut de paiment selon l'âge**

In [43]:
plot_data = df_train.copy()
plot_data["Défaut"] = plot_data["default_payment_next_month"].map(
    {
        0: "Pas de défaut",
        1: "Défaut de paiement",
    }
)

fig = go.Figure()

for default_status in ["Pas de défaut", "Défaut de paiement"]:
    data = plot_data[plot_data["Défaut"] == default_status]

    fig.add_trace(
        go.Box(
            x=data["Défaut"],
            y=data["age"],
            name=default_status,
            boxpoints="all",
            jitter=0.35,
            pointpos=0,
            marker={"size": 5, "opacity": 0.45},
            line={"width": 1.5},
            showlegend=False,
        )
    )

fig.update_layout(
    title="Âge des clients selon le défaut de paiement",
    xaxis_title="Statut de paiement",
    yaxis_title="Âge",
    template="plotly_white",
    height=500,
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

nous avons une distribution assez equivalente des âges dans les 2 modalités de la varible `default_payment_next_month`, même si nous pouvons dire que les clientts avec un defaut de paiement sont legerement plus agé en moyenne que ceux qui n'ont pas de defaut de paiement mais cela reste negligeable.

In [44]:
pay_columns = ["pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"]

pay_data = df_train[pay_columns + ["default_payment_next_month"]].melt(
    id_vars="default_payment_next_month",
    value_vars=pay_columns,
    var_name="Variable pay",
    value_name="Modalité de remboursement",
)

pay_default_rate = (
    pay_data.groupby(["Variable pay", "Modalité de remboursement"])
    ["default_payment_next_month"]
    .mean()
    .reset_index(name="Taux de défaut")
)
pay_default_rate["Taux de défaut (%)"] = pay_default_rate["Taux de défaut"] * 100

fig = px.bar(
    pay_default_rate,
    x="Modalité de remboursement",
    y="Taux de défaut (%)",
    color="Variable pay",
    barmode="group",
    text="Taux de défaut (%)",
    title="Taux de défaut selon les modalités des variables pay_*",
    labels={
        "Modalité de remboursement": "Modalité de remboursement",
        "Taux de défaut (%)": "Taux de défaut (%)",
        "Variable pay": "Mois observé",
    },
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{text:.1f} %",
    textposition="outside",
)

fig.update_layout(
    height=650,
    yaxis={"range": [0, max(pay_default_rate["Taux de défaut (%)"]) * 1.2]},
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

les pourcentages de defaut de carte de credit sont plus elevé dans les modalités presentant les retards de paiements ce qui est logique.

In [45]:
bill_columns = [f"bill_amt_{i}" for i in range(1, 7)]
default_labels = {
    0: "Pas de défaut",
    1: "Défaut de paiement",
}

default_order = ["Pas de défaut", "Défaut de paiement"]

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=bill_columns,
    shared_yaxes=False,
    vertical_spacing=0.18,
    horizontal_spacing=0.08,
)

for index, column in enumerate(bill_columns):
    row = index // 3 + 1
    col = index % 3 + 1

    for default_status in [0, 1]:
        data = df_train[df_train["default_payment_next_month"] == default_status]

        fig.add_trace(
            go.Box(
                x=[default_labels[default_status]] * len(data),
                y=data[column],
                name=default_labels[default_status],
                boxmean=True,
                boxpoints=False,
                legendgroup=default_labels[default_status],
                showlegend=(index == 0),
            ),
            row=row,
            col=col,
        )

    fig.update_xaxes(
        title_text="Statut de paiement",
        categoryorder="array",
        categoryarray=default_order,
        row=row,
        col=col,
    )
    fig.update_yaxes(
        title_text="Montant de la facture",
        row=row,
        col=col,
    )

fig.update_layout(
    title="Répartition des montants facturés selon le défaut de paiement",
    template="plotly_white",
    height=850,
    width=1200,
    boxmode="group",
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

Les distributions de `bill_amt_1` à `bill_amt_6` sont très proches entre clients en défaut et sans défaut (écart de moyennes entre -1,4 % et +3,2 % selon le mois, négligeable vu la forte dispersion). Contrairement aux variables `pay_*`, le montant facturé seul n'est donc pas discriminant : c'est le comportement de remboursement (retard), pas le montant de la facture, qui semble lié au risque de défaut.

In [46]:
pay_amt_columns = [f"pay_amt_{i}" for i in range(1, 7)]
default_labels = {
    0: "Pas de défaut",
    1: "Défaut de paiement",
}

default_order = ["Pas de défaut", "Défaut de paiement"]

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=pay_amt_columns,
    shared_yaxes=False,
    vertical_spacing=0.18,
    horizontal_spacing=0.08,
)

for index, column in enumerate(pay_amt_columns):
    row = index // 3 + 1
    col = index % 3 + 1

    for default_status in [0, 1]:
        data = df_train[df_train["default_payment_next_month"] == default_status]

        fig.add_trace(
            go.Box(
                x=[default_labels[default_status]] * len(data),
                y=data[column],
                name=default_labels[default_status],
                boxmean=True,
                boxpoints=False,
                legendgroup=default_labels[default_status],
                showlegend=(index == 0),
            ),
            row=row,
            col=col,
        )

    fig.update_xaxes(
        title_text="Statut de paiement",
        categoryorder="array",
        categoryarray=default_order,
        row=row,
        col=col,
    )
    fig.update_yaxes(
        title_text="Montant payé",
        row=row,
        col=col,
    )

fig.update_layout(
    title="Répartition des montants payés selon le défaut de paiement",
    template="plotly_white",
    height=850,
    width=1200,
    boxmode="group",
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

Contrairement aux `bill_amt_*`, les `pay_amt_*` montrent une nette différence entre les deux groupes : les clients en défaut paient en moyenne **36 % à 60 % de moins** que les clients sans défaut (et médianes systématiquement plus basses), selon le mois. Les boîtes des clients en défaut sont plus basses et plus resserrées près de 0.

Ce montant payé est donc un signal plus discriminant que le montant facturé : ce n'est pas la taille de la facture qui distingue les défauts, mais le fait de rembourser peu par rapport à ce qui est dû — cohérent avec le lien déjà observé entre retards (`pay_*`) et défaut.

### 3.4 Matrice des corrélations

In [47]:
corr_matrix = df_train[numeric_columns].corr().round(2)

fig = go.Figure(
    data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        colorscale="RdBu_r",
        zmid=0,
        zmin=-1,
        zmax=1,
        text=corr_matrix.values,
        texttemplate="%{text}",
        textfont={"size": 9},
        colorbar={"title": "Corrélation"},
    )
)

fig.update_layout(
    title="Matrice de corrélation des variables numériques",
    template="plotly_white",
    height=800,
    width=900,
    xaxis={"tickangle": -45},
)

from IPython.display import HTML, display

display(HTML(fig.to_html(include_plotlyjs="cdn")))

##### **Analyse de la matrice de corrélation**

- **Corrélation avec la cible** : les `pay_*` sont de loin les variables les plus corrélées à `default_payment_next_month` (0,25 à 0,38, `pay_0` en tête), ce qui confirme le lien déjà observé entre retards et défaut. `limit_balance` a une corrélation négative modérée (-0,17) : les clients à plafond élevé font moins souvent défaut. Les `pay_amt_*` sont faiblement corrélées (-0,05 à -0,09) et les `bill_amt_*` quasiment nulles (proches de 0), cohérent avec les constats des boxplots précédents.

- **Multicolinéarité forte entre les `bill_amt_*`** : les factures de mois consécutifs sont très corrélées entre elles (0,80 à 0,95, ex. `bill_amt_5`/`bill_amt_6` = 0,95). C'est logique (le solde évolue peu d'un mois à l'autre) mais c'est un point de vigilance pour la modélisation : utiliser les 6 variables telles quelles dans un modèle linéaire apporterait beaucoup de redondance. Il faudra envisager d'en garder un sous-ensemble, une agrégation (moyenne, tendance) ou une réduction de dimension.

- Les `pay_*` sont eux aussi corrélés entre eux (jusqu'à ~0,80 pour les mois consécutifs), ce qui traduit un comportement de retard qui persiste dans le temps plutôt que ponctuel.